In [1]:
import pandas as pd

from util import *

rand_state = rand_seed()
print(f"Using random state: {rand_state}")

Using random state: 721


In [2]:
x_cols = ["age", "gender", "location", "size"]
y_col = "mdm2"
TEST_SIZE = 0.2

df = load_with_columns(x_cols + [y_col])
df.to_csv("data/specific.csv", index=False)

Datapoints: 2706 --> 1657 (1049 removed)


In [3]:
df = df[df["size"] > 10]
print(df.shape)
x = df.loc[:, x_cols]
y = df[y_col]

x["gender"] = x["gender"].map({"m": 0, "f": 1})
x = pd.get_dummies(x)
y = y.astype(int)
x

(1501, 5)


,age,gender,size,location_hn,location_lld,location_lls,location_o,location_t,location_uld,location_uls
0,74.0,1,20.0,False,False,False,True,False,False,False
1,45.0,1,71.0,True,False,False,False,False,False,False
2,73.0,1,15.0,False,False,False,False,False,False,True
3,79.0,0,14.0,False,False,False,False,False,True,False
4,68.0,1,159.0,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...
1652,59.0,0,145.0,False,True,False,False,False,False,False
1653,52.0,0,90.0,False,False,False,False,True,False,False
1654,52.0,1,25.0,False,False,False,False,False,True,False
1655,81.0,1,75.0,False,True,False,False,False,False,False


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

### Prepare for fit
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=TEST_SIZE,
    random_state=rand_state,
    stratify=y,
)
x_train, y_train = SMOTE(
    sampling_strategy=0.5,
    random_state=rand_state,
).fit_resample(x_train, y_train)

scaler = StandardScaler()

x_train = pd.DataFrame(scaler.fit_transform(x_train), columns=x_train.columns, index=x_train.index)
x_test = pd.DataFrame(scaler.fit_transform(x_test), columns=x_test.columns, index=x_test.index)

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader


class DataFrameDataset(Dataset):
    def __init__(self, features, target):
        # Convert to tensors
        self.features = torch.FloatTensor(features.values if isinstance(features, pd.DataFrame) else features)
        self.target = torch.FloatTensor(target.values if isinstance(target, pd.Series) else target)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.target[idx]

train_dataset = DataFrameDataset(x_train, y_train)
val_dataset = DataFrameDataset(x_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

In [6]:
from torch import nn, optim
device = torch.device("mps")

model = nn.Sequential(
    nn.Linear(10, 256),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(128, 32),
    nn.ReLU(),

    nn.Linear(32, 1),
    nn.Sigmoid()
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

In [7]:
from sklearn.metrics import confusion_matrix
import torch

epochs = 200
min_val_loss = float('inf')
patience = 15
patience_counter = 0

train_losses = []
val_losses = []

for e in range(epochs):
    model.train()
    train_loss = 0.0
    train_samples = 0

    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)

        optimizer.zero_grad()
        yhat = model(bx).squeeze()

        # Handle single sample batches
        if yhat.dim() == 0:
            yhat = yhat.unsqueeze(0)
        if by.dim() == 0:
            by = by.unsqueeze(0)

        loss = criterion(yhat, by)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * bx.size(0)
        train_samples += bx.size(0)

    # Calculate average training loss
    avg_train_loss = train_loss / train_samples

    model.eval()

    all_predictions = []
    all_targets = []
    test_loss = 0.0
    test_samples = 0

    with torch.no_grad():
        for batch_features, batch_target in test_loader:
            batch_features, batch_target = batch_features.to(device), batch_target.to(device)

            # Get model outputs
            outputs = model(batch_features).squeeze()

            # Handle single sample batches
            if outputs.dim() == 0:
                outputs = outputs.unsqueeze(0)
            if batch_target.dim() == 0:
                batch_target = batch_target.unsqueeze(0)

            # Calculate loss
            loss = criterion(outputs, batch_target)
            test_loss += loss.item() * batch_features.size(0)
            test_samples += batch_features.size(0)

            # Get predictions - assuming binary classification with sigmoid
            if hasattr(criterion, '__class__') and 'BCE' in criterion.__class__.__name__:
                # Binary classification with BCEWithLogitsLoss
                y_pred_batch = torch.sigmoid(outputs) > 0.5
            else:
                # Multi-class classification with CrossEntropyLoss
                _, y_pred_batch = torch.max(outputs, 1)

            # Store predictions and targets
            all_predictions.extend(y_pred_batch.cpu().numpy().astype(int))
            all_targets.extend(batch_target.cpu().numpy().astype(int))

    # Calculate average test loss
    avg_val_loss = test_loss / test_samples

    # Store losses
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    # Create confusion matrix following your pattern
    y_test = all_targets
    y_pred = all_predictions
    cnf = confusion_matrix(y_test, y_pred)

    print(f"Epoch {e+1:03d} | Train loss: {avg_train_loss:.4f}, Test loss: {avg_val_loss:.4f}")
    summarise_cnf(cnf)



Epoch 001 | Train loss: 0.7880, Test loss: 0.7877
Sensitivity: 1.0
Specificity: 0.0
Epoch 002 | Train loss: 0.7101, Test loss: 0.6997
Sensitivity: 1.0
Specificity: 0.0
Epoch 003 | Train loss: 0.6902, Test loss: 0.6978
Sensitivity: 1.0
Specificity: 0.0
Epoch 004 | Train loss: 0.6761, Test loss: 0.7357
Sensitivity: 1.0
Specificity: 0.0
Epoch 005 | Train loss: 0.6569, Test loss: 0.7199
Sensitivity: 1.0
Specificity: 0.0
Epoch 006 | Train loss: 0.6411, Test loss: 0.7277
Sensitivity: 1.0
Specificity: 0.0
Epoch 007 | Train loss: 0.6294, Test loss: 0.7335
Sensitivity: 1.0
Specificity: 0.0
Epoch 008 | Train loss: 0.6215, Test loss: 0.7240
Sensitivity: 1.0
Specificity: 0.0
Epoch 009 | Train loss: 0.6209, Test loss: 0.7353
Sensitivity: 1.0
Specificity: 0.0
Epoch 010 | Train loss: 0.6174, Test loss: 0.7215
Sensitivity: 1.0
Specificity: 0.0076045627376425855
Epoch 011 | Train loss: 0.6179, Test loss: 0.7214
Sensitivity: 1.0
Specificity: 0.015209125475285171
Epoch 012 | Train loss: 0.6166, Test loss